<a href="https://colab.research.google.com/github/freida20git/bird-detection-tracking/blob/main/func_ocsort.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalations:

In [ ]:
import sys
sys.path.append('/content/OC_SORT') # Add the parent directory to sys.path
from trackers.ocsort_tracker.ocsort import OCSort
tracker = OCSort(det_thresh=0.5)
print("OCSort loaded successfully!")

OCSort loaded successfully!


Since OC-SORT assigns detections to tracks internally (via Hungarian algorithm + IOU), we re-match each track to its most likely original detection after tracking, based on spatial proximity.

To attach the most relevant confidence score to each tracked object in each frame:

In [ ]:
def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)

In [ ]:
import cv2
import numpy as np
import json
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Run OcSORT

In [ ]:
'''default_tracker = OCSort(
    det_thresh=0.4,
    max_age=50,
    min_hits=4,
    iou_threshold=0.3,
    delta_t=3,
    inertia=0.4,
    asso_func="iou"
)
'''
#default_tracker = OCSort(det_thresh= 0.5, max_age = 50, min_hits = 2, iou_threshold = 0.3, delta_t = 3, inertia = 0.4, asso_func = "ciou")

# best parameters after trying 14 combinations + gridsearch with sklearn:
default_tracker = OCSort(det_thresh= 0.4, max_age = 90, min_hits = 2, iou_threshold = 0.3, delta_t = 3, inertia = 0.3, asso_func = "ciou")

def run_ocsort_save_json(model_pt_file, video_path, output_video_path, json_output_name, my_tracker=default_tracker):
    model = model_pt_file
    tracking_results = []

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    tracker = my_tracker
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame)
        detections = results[0].boxes

        if detections is not None and len(detections) > 0:
            boxes = detections.xyxy.cpu().numpy()
            scores = detections.conf.cpu().numpy()
            dets = np.hstack((boxes, scores.reshape(-1, 1)))

            detection_map = [{
                "bbox": dets[i][:4],
                "confidence": float(dets[i][4]),
                "used": False
            } for i in range(len(dets))]


        else:
            dets = np.empty((0, 5))
            detection_map = []

        tracks = tracker.update(dets, frame.shape[:2], frame.shape[:2])
        frame_objects = []

        for track in tracks:
            x1, y1, x2, y2, track_id = track
            track_bbox = [x1, y1, x2, y2]
            track_center = bbox_center(track_bbox)

            # Match with closest unused detection by center distance
            closest_det = None
            min_dist = float("inf")
            for det in detection_map:
                if det["used"]:
                    continue
                det_center = bbox_center(det["bbox"])
                dist = (track_center[0] - det_center[0])**2 + (track_center[1] - det_center[1])**2
                if dist < min_dist:
                    min_dist = dist
                    closest_det = det

            confidence = None
            if closest_det:
                confidence = closest_det["confidence"]
                closest_det["used"] = True  # Mark as used

            # Draw
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 255, 255), 2)
            cv2.putText(frame, f'ID {int(track_id)}', (int(x1), int(y1) - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            frame_objects.append({
                "track_id": int(track_id),
                "class_id": 0,
                "class_name": "bird",
                "confidence": confidence,
                "bbox": {
                    "x1": int(x1),
                    "y1": int(y1),
                    "x2": int(x2),
                    "y2": int(y2)
                }
            })

        if frame_objects:
            tracking_results.append({
                "frame_number": frame_id + 1,
                "objects": frame_objects
            })

        out.write(frame)
        frame_id += 1

    cap.release()
    out.release()

    with open(json_output_name, "w") as f:
        json.dump(tracking_results, f, indent=2)

    print(f"✅ Done! Output saved to {output_video_path} and {json_output_name}")


In [ ]:
# usage example:
'''
model = YOLO("/content/bestbirdsonly.pt")
run_ocsort_save_json(model, "birds_sky.mp4", "output_birds_sky.mp4", "birds_sky_results.json")
or(send ocsort trackjer parameters):
run_ocsort_save_json(model, "/content/similar_birds.mp4", "output_similar_birds.mp4", "similar_birds_results.json",  OCSort(
    det_thresh=0.4,
    max_age=50,
    min_hits=4,
    iou_threshold=0.3,
    delta_t=3,
    inertia=0.4,
    asso_func="iou"
))
'''
